# Borzoi / Flashzoi: Prediction + Benchmarking

Этот ноутбук объединяет:

- предсказание экспрессии из `run_borzoi_prediction_pytorch.py`
- benchmark / корреляции / графики из `only_benchmarking_clean.ipynb`

В notebook теперь есть переключатель:

- `model_variant = "borzoi"`
- `model_variant = "flashzoi"`

Логика разбита на шаги:

1. конфигурация и пути
2. загрузка модели и аннотаций
3. функции для инференса по генам
4. запуск предсказания и сохранение `csv`
5. benchmark против `ground truth`
6. визуализация по таргетам


In [ ]:
import os
import sys
from contextlib import nullcontext
from typing import Optional

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from tqdm import tqdm
from pyfaidx import Fasta
from borzoi_pytorch import Borzoi, Transcriptome

# ----------------------------
# Основная конфигурация
# ----------------------------
split = "valid"  # "test" или "valid"
model_variant = "borzoi"  # "borzoi" или "flashzoi"
replicate = 0
run_prediction = True

# Если True, перед агрегацией по экзонам предсказания будут переведены
# из squashed scale обратно в coverage-like scale по параметрам из targets_human.txt.
undo_track_transform = True

# Что использовать как ground truth для benchmark-а:
# "main"     -> {split}_true_human.csv
# "tcell_bw" -> borzoi_ground_true_from_bw_T-cell.csv
true_source = "main"

# Если хочешь ограничить число строк при отладке, поставь integer.
debug_max_rows = None

# Если хочешь рисовать только часть таргетов, задай список id.
plot_targets = None

# ----------------------------
# Пути к данным
# ----------------------------
BORZOI_DIR = "/home/jovyan/shares/SR003.nfs2/aspeedok/GENA_LM/downstream_tasks/expression_prediction/benchmarks/borzoi"

FASTA_PATH = "/home/jovyan/shares/SR003.nfs2/aspeedok/GENA_LM/downstream_tasks/expression_prediction/datasets/data/genomes/hg38/hg38.fa"
TARGETS_FILE = os.path.join(BORZOI_DIR, "targets_human.txt")
ANNOT_GTF = os.path.join(BORZOI_DIR, "gencode.v29.primary_assembly.annotation_UCSC_names.gtf.gz")
MAPPING_PATH = "/home/jovyan/shares/SR003.nfs2/aspeedok/GENA_LM/downstream_tasks/expression_prediction/datasets/data/file_mappings/Expression_dataset_v1_csv_file_mappings.csv"
# SELECTED_TARGETS_PATH = os.path.join(BORZOI_DIR, "selected_targets.csv")

genes_forward_path = os.path.join(BORZOI_DIR, f"human.{split}.forward.csv")
genes_reverse_path = os.path.join(BORZOI_DIR, f"human.{split}.reverse.csv")

true_main_path = os.path.join(BORZOI_DIR, f"{split}_true_human.csv")
# true_tcell_bw_path = "/data/ddpanchenko/main_dom/GENA/Borzoi/predictions/22012026/borzoi_ground_true_from_bw_T-cell.csv"
SCORE_SRC_DIR = "/home/jovyan/shares/SR003.nfs2/aspeedok/GENA_LM/downstream_tasks/expression_prediction/datasets/src"

# ----------------------------
# Параметры модели / устройства
# ----------------------------
SEQ_LEN = 524_288
MODEL_STRIDE = 32

# Это рабочая гипотеза для привязки координат PyTorch-выхода к геному.
# Значение взято из upstream notebook borzoi-pytorch, где TF-предсказания режут
# как [..., 5104:-5104], чтобы получить тот же центральный выход, что и в PyTorch.
# Для Flashzoi пока используем ту же привязку, но это место стоит валидировать отдельно.
MODEL_CROP_BINS = 5120

# Sanity check: ожидаем длину выхода PyTorch Borzoi/Flashzoi в 6144 бинов.
EXPECTED_OUTPUT_BINS = 6144

DEVICE = torch.device("cuda:6" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

model_name_map = {
    "borzoi": f"johahi/borzoi-replicate-{replicate}",
    "flashzoi": f"johahi/flashzoi-replicate-{replicate}",
}
if model_variant not in model_name_map:
    raise ValueError(f"Unknown model_variant: {model_variant}")

model_name = model_name_map[model_variant]
use_flashzoi = model_variant == "flashzoi"
use_autocast = use_flashzoi

if use_flashzoi and DEVICE.type != "cuda":
    raise RuntimeError(
        "Flashzoi в upstream README требует modern Nvidia GPU и autocast. "
        "На CPU в этом notebook он отключён."
    )

if DEVICE.type == "cuda":
    autocast_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
else:
    autocast_dtype = None

pred_out_path = os.path.join(
    BORZOI_DIR,
    f"{model_variant}_predictions_{split}_replicate{replicate}_pytorch.csv",
)

print("Model variant:", model_variant)
print("Model name:", model_name)
print("Autocast enabled:", use_autocast)
print("Autocast dtype:", autocast_dtype)
print("Undo track transform:", undo_track_transform)


ModuleNotFoundError: No module named 'borzoi_pytorch'

In [1]:
# Широкая qnorm ground truth: столбцы = id (ENCFF) из обоих маппингов _qnorm.
# Один и тот же id (ENCFF файла TPM): приоритет строки из file_mappings_borzoi_human_qnorm (keep="last").

FILE_MAPPINGS_DIR = os.path.normpath(
    os.path.join(BORZOI_DIR, "..", "..", "datasets", "data", "file_mappings")
)
EXPR_MAP_QNORM = os.path.join(FILE_MAPPINGS_DIR, "Expression_dataset_v1_csv_file_mappings_qnorm.csv")
BORZOI_MAP_QNORM = os.path.join(FILE_MAPPINGS_DIR, "file_mappings_borzoi_human_qnorm.csv")

expr_map = pd.read_csv(EXPR_MAP_QNORM)
borzoi_map = pd.read_csv(BORZOI_MAP_QNORM)
samples_meta = pd.concat([expr_map, borzoi_map], ignore_index=True).drop_duplicates(
    subset=["id"], keep="last"
)


def resolve_tpm_path(csv_rel: str) -> str:
    return os.path.normpath(os.path.join(FILE_MAPPINGS_DIR, csv_rel))


out_true_wide_path = os.path.join(BORZOI_DIR, f"{split}_true_human_all_qnorm_id.csv")

series_list = []
missing_files = []
for _, row in tqdm(samples_meta.iterrows(), total=len(samples_meta), desc="Load qnorm TPM"):
    tpm_path = resolve_tpm_path(row["csv"])
    if not os.path.isfile(tpm_path):
        missing_files.append(tpm_path)
        continue
    wide = pd.read_csv(tpm_path, nrows=1)
    col_name = str(row["id"])
    ser = pd.Series(wide.iloc[0].to_numpy(), index=wide.columns.astype(str), name=col_name)
    series_list.append(ser)

if missing_files:
    raise FileNotFoundError(
        f"Нет {len(missing_files)} TPM (первые 5): {missing_files[:5]}"
    )

wide_true_df = pd.concat(series_list, axis=1, join="outer")
wide_true_df.insert(0, "gene_id", wide_true_df.index.to_numpy())
wide_true_df = wide_true_df.reset_index(drop=True)

# Ограничить гены сплита как в benchmark (число строк как у valid_true_human): поставь True
restrict_to_split_genes = False
if restrict_to_split_genes:
    bw_f = pd.read_csv(genes_forward_path, sep="\t")
    bw_r = pd.read_csv(genes_reverse_path, sep="\t")
    g_ok = set(pd.concat([bw_f, bw_r], ignore_index=True)["gene_id"].astype(str))
    wide_true_df = wide_true_df[wide_true_df["gene_id"].astype(str).isin(g_ok)].reset_index(drop=True)

wide_true_df.to_csv(out_true_wide_path, index=False)
print("Saved", wide_true_df.shape, "->", out_true_wide_path)

NameError: name 'os' is not defined

In [14]:
genome = Fasta(FASTA_PATH)

map_df = pd.read_csv(MAPPING_PATH)
selected_targets_df = map_df[["id", "original_id"]].drop_duplicates().copy()
selected_original_ids = set(selected_targets_df["original_id"])

targets_df = pd.read_csv(TARGETS_FILE, sep="\t", index_col=0)
targets_df["identifier_base"] = targets_df["identifier"].str.replace(r"[+-]$", "", regex=True)
targets_df["file_acc"] = targets_df["file"].str.extract(r'(ENC(?:SR|FF)[0-9A-Z]+)')
targets_df["original_id"] = np.where(
    targets_df["file_acc"].isin(selected_original_ids),
    targets_df["file_acc"],
    np.where(
        targets_df["identifier_base"].isin(selected_original_ids),
        targets_df["identifier_base"],
        pd.NA,
    ),
)
targets_df_sub = (
    targets_df.dropna(subset=["original_id"])
    .merge(selected_targets_df, on="original_id", how="inner", validate="many_to_one")
    .copy()
)

target_index_sub = targets_df_sub.index.to_numpy(dtype=int)
encff_ids = targets_df_sub["identifier"].tolist()

# Параметры inverse transform берём напрямую из targets_human.txt.
target_scale = targets_df_sub["scale"].to_numpy(dtype=np.float32)
target_clip = targets_df_sub["clip"].to_numpy(dtype=np.float32)
target_clip_soft = targets_df_sub["clip_soft"].to_numpy(dtype=np.float32)
target_sum_stat = targets_df_sub["sum_stat"].astype(str).to_numpy()
target_transform = np.where(target_sum_stat == "sum_sqrt", 3.0 / 4.0, 1.0).astype(np.float32)

target_params_df = targets_df_sub[
    ["identifier", "id", "original_id", "clip", "clip_soft", "scale", "sum_stat", "description"]
].copy()

print("Selected target tracks:", len(target_index_sub))
print("Selected benchmark targets:", targets_df_sub["id"].nunique())
print("Unique target transform configs:")
print(
    target_params_df[["clip", "clip_soft", "scale", "sum_stat"]]
    .drop_duplicates()
    .sort_values(["clip", "clip_soft", "scale", "sum_stat"])
    .to_string(index=False)
)


Selected targets: 26
Unique target transform configs:
 clip  clip_soft  scale sum_stat
  768        384    0.3 sum_sqrt


In [15]:
if DEVICE.type == "cuda":
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False

model = Borzoi.from_pretrained(model_name)
model.to(DEVICE)
model.eval()

print(f"Loaded {model_name} on {DEVICE}")

Loaded johahi/borzoi-replicate-0 on cuda:6


In [16]:
transcriptome_by_id = Transcriptome(ANNOT_GTF, use_geneid=True)
transcriptome_by_name = Transcriptome(ANNOT_GTF, use_geneid=False)

In [5]:
transcriptome_by_name.genes

{'DDX11L1': <borzoi_pytorch.gene_utils.Gene at 0x7f033f789850>,
 'WASH7P': <borzoi_pytorch.gene_utils.Gene at 0x7f033f78a6d0>,
 'MIR6859-1': <borzoi_pytorch.gene_utils.Gene at 0x7f033f78b410>,
 'MIR1302-2HG': <borzoi_pytorch.gene_utils.Gene at 0x7f033f78a990>,
 'MIR1302-2': <borzoi_pytorch.gene_utils.Gene at 0x7f033f78b650>,
 'FAM138A': <borzoi_pytorch.gene_utils.Gene at 0x7f033f78b6d0>,
 'OR4G4P': <borzoi_pytorch.gene_utils.Gene at 0x7f033f791dd0>,
 'OR4G11P': <borzoi_pytorch.gene_utils.Gene at 0x7f033f7915d0>,
 'OR4F5': <borzoi_pytorch.gene_utils.Gene at 0x7f033f792d10>,
 'AL627309.1': <borzoi_pytorch.gene_utils.Gene at 0x7f033f793810>,
 'AL627309.3': <borzoi_pytorch.gene_utils.Gene at 0x7f033f793a50>,
 'CICP27': <borzoi_pytorch.gene_utils.Gene at 0x7f033f7a0c90>,
 'AL627309.6': <borzoi_pytorch.gene_utils.Gene at 0x7f033f793b10>,
 'AL627309.7': <borzoi_pytorch.gene_utils.Gene at 0x7f033f7a1590>,
 'AL627309.2': <borzoi_pytorch.gene_utils.Gene at 0x7f033f7a2010>,
 'AL627309.5': <borzoi

In [6]:
transcriptome_by_id.genes

{'ENSG00000223972.5': <borzoi_pytorch.gene_utils.Gene at 0x7f05e2fcc0d0>,
 'ENSG00000227232.5': <borzoi_pytorch.gene_utils.Gene at 0x7f062d742d90>,
 'ENSG00000278267.1': <borzoi_pytorch.gene_utils.Gene at 0x7f05e2fcfb50>,
 'ENSG00000243485.5': <borzoi_pytorch.gene_utils.Gene at 0x7f05e43dee50>,
 'ENSG00000284332.1': <borzoi_pytorch.gene_utils.Gene at 0x7f05e2fd0590>,
 'ENSG00000237613.2': <borzoi_pytorch.gene_utils.Gene at 0x7f05e2fd2f90>,
 'ENSG00000268020.3': <borzoi_pytorch.gene_utils.Gene at 0x7f05e2fbed10>,
 'ENSG00000240361.2': <borzoi_pytorch.gene_utils.Gene at 0x7f05e2fbee10>,
 'ENSG00000186092.6': <borzoi_pytorch.gene_utils.Gene at 0x7f05e500eb10>,
 'ENSG00000238009.6': <borzoi_pytorch.gene_utils.Gene at 0x7f070afc2190>,
 'ENSG00000239945.1': <borzoi_pytorch.gene_utils.Gene at 0x7f05e2fc7590>,
 'ENSG00000233750.3': <borzoi_pytorch.gene_utils.Gene at 0x7f05e2fc6890>,
 'ENSG00000268903.1': <borzoi_pytorch.gene_utils.Gene at 0x7f05e2fc7a90>,
 'ENSG00000269981.1': <borzoi_pytorch.

In [9]:
if 'AC016700' in transcriptome_by_name.genes:
    print('yes')
else:
    print('no')
 

no


In [21]:
def one_hot_encode_dna(seq: str) -> np.ndarray:
    mapping = {
        "A": [1, 0, 0, 0],
        "C": [0, 1, 0, 0],
        "G": [0, 0, 1, 0],
        "T": [0, 0, 0, 1],
        "N": [0, 0, 0, 0],
    }
    arr = np.zeros((len(seq), 4), dtype="float32")
    for i, base in enumerate(seq.upper()):
        arr[i] = mapping.get(base, mapping["N"])
    return arr


def get_window_sequence(row, seq_len=SEQ_LEN):
    chrom = row["chrom"]
    tss = int(row["TSS"])
    strand = row["gene_strand"]

    half = seq_len // 2
    start = tss - half
    if start < 1:
        start = 1
    end = start + seq_len

    seq = genome[chrom][start:end].seq
    if len(seq) < seq_len:
        seq = seq + "N" * (seq_len - len(seq))
    elif len(seq) > seq_len:
        seq = seq[:seq_len]

    if strand == "-":
        comp = str.maketrans("ACGTacgt", "TGCAtgca")
        seq = seq.translate(comp)[::-1]

    return seq, start


p = 0


def find_gene_obj(row):
    global p

    gid = str(row["gene_id_unversioned"])
    if gid in transcriptome_by_id.genes:
        return transcriptome_by_id.genes[gid]

    gname = str(row["gene_name"])
    if gname in transcriptome_by_name.genes:
        return transcriptome_by_name.genes[gname]

    p += 1
    raise ValueError(f"Gene not found in transcriptome: {gid} / {gname}")


def get_autocast_context():
    if use_autocast and DEVICE.type == "cuda":
        return torch.autocast(device_type="cuda", dtype=autocast_dtype)
    return nullcontext()


def undo_track_transform_from_targets(pred_2d: np.ndarray) -> np.ndarray:
    # pred_2d: [n_bins, n_targets_sub] в squashed scale
    x = pred_2d.astype(np.float32).copy()

    # 1. Убираем track-specific scale
    x = x / target_scale[None, :]

    # 2. Убираем soft squash по track-specific clip_soft
    clip_soft = target_clip_soft[None, :]
    mask = x > clip_soft
    x = np.where(mask, (x - clip_soft) ** 2 + clip_soft, x)

    # 3. Обращаем степень:
    #    sum_sqrt -> transform 3/4 -> inverse 4/3
    #    sum      -> transform 1.0 -> inverse 1.0
    x = x ** (1.0 / target_transform[None, :])
    return x


def predict_one_sequence(one_hot_seq: np.ndarray) -> np.ndarray:
    x = torch.from_numpy(one_hot_seq).permute(1, 0).unsqueeze(0).to(DEVICE)

    with torch.inference_mode():
        with get_autocast_context():
            y = model(x)

    pred = y[0].detach().float().cpu().numpy().transpose(1, 0)

    if pred.shape[0] != EXPECTED_OUTPUT_BINS:
        raise ValueError(
            f"Unexpected number of output bins: {pred.shape[0]} "
            f"(expected {EXPECTED_OUTPUT_BINS})"
        )

    pred = pred[:, target_index_sub]

    if undo_track_transform:
        pred = undo_track_transform_from_targets(pred)

    return pred


def gene_vector_for_row(row) -> Optional[np.ndarray]:
    seq, seq_start = get_window_sequence(row)
    one_hot = one_hot_encode_dna(seq)
    pred = predict_one_sequence(one_hot)

    if row["gene_strand"] == "-":
        pred = pred[::-1, :]

    gene_obj = find_gene_obj(row)

    seq_out_start = seq_start + MODEL_STRIDE * MODEL_CROP_BINS
    seq_out_len = MODEL_STRIDE * pred.shape[0]

    gene_slice = gene_obj.output_slice(
        seq_out_start,
        seq_out_len,
        MODEL_STRIDE,
        False,
    )

    if len(gene_slice) == 0:
        print(f"Gene has empty output slice: {row['gene_id']}")
        return None

    # В этом notebook считаем средний сигнал по экзонным бинам proxy для gene expression.
    exon_pred = pred[gene_slice, :]
    expr = exon_pred.mean(axis=0)

    return expr.astype(np.float32)


def run_split(forward_path, reverse_path, out_csv_path, max_rows=None):
    print(f"\n=== Run split ===\n{forward_path}\n{reverse_path}\n=> {out_csv_path}")

    df_f = pd.read_csv(forward_path, sep="\t")
    df_f = df_f
    df_r = pd.read_csv(reverse_path, sep="\t")
    df_r = df_r

    df = pd.concat([df_f, df_r], ignore_index=True)
    if max_rows is not None:
        df = df.iloc[:max_rows].copy()

    print("Total genes (forward+reverse):", len(df))

    gene_to_vecs = {}

    for idx, row in tqdm(df.iterrows(), total=len(df)):
        try:
            vec = gene_vector_for_row(row)
        except Exception as e:
            print(f"  [WARN] пропускаем строку {idx} ({row['gene_id']}): {e}")
            continue

        if vec is None:
            continue

        gid = str(row["gene_id"])
        gene_to_vecs.setdefault(gid, []).append(vec)

    genes = []
    preds = []
    for gid, vec_list in gene_to_vecs.items():
        arr = np.stack(vec_list, axis=0)
        mean_vec = arr.mean(axis=0)
        genes.append(gid)
        preds.append(mean_vec)

    preds = np.stack(preds, axis=0)

    out_df = pd.DataFrame(preds, columns=encff_ids)
    out_df.insert(0, "gene_id", genes)

    merged = out_df.melt(
        id_vars=["gene_id"],
        value_vars=encff_ids,
        var_name="identifier",
        value_name="expr",
    )

    merged = merged.merge(
        targets_df_sub[["identifier", "id"]],
        on="identifier",
        how="left",
    )

    final = merged.groupby(["gene_id", "id"])["expr"].sum().reset_index()
    final_df = final.pivot(index="gene_id", columns="id", values="expr").reset_index()
    final_df.to_csv(out_csv_path, index=False)

    print("Done:", final_df.shape)
    print("Missing genes count:", p)
    return final_df


## Шаг 1. Предсказание PyTorch Borzoi / Flashzoi

Если `run_prediction = True`, ячейка ниже прогонит выбранную модель по всем генам
и сохранит результат в `pred_out_path`.

Варианты:

- `model_variant = "borzoi"`: обычный PyTorch-порт Borzoi
- `model_variant = "flashzoi"`: ускоренная версия; в этом notebook для неё автоматически включается `autocast`

Если `run_prediction = False`, ячейка просто загрузит уже сохранённый `csv`.


In [22]:
if run_prediction:
    pred_gene_df = run_split(
        genes_forward_path,
        genes_reverse_path,
        pred_out_path,
        max_rows=debug_max_rows,
    )
else:
    pred_gene_df = pd.read_csv(pred_out_path)

print(pred_out_path)
print("p =", p)
pred_gene_df.head()



=== Run split ===
/home/jovyan/shares/SR003.nfs2/aspeedok/GENA_LM/downstream_tasks/expression_prediction/benchmarks/borzoi/human.valid.forward.csv
/home/jovyan/shares/SR003.nfs2/aspeedok/GENA_LM/downstream_tasks/expression_prediction/benchmarks/borzoi/human.valid.reverse.csv
=> /home/jovyan/shares/SR003.nfs2/aspeedok/GENA_LM/downstream_tasks/expression_prediction/benchmarks/borzoi/borzoi_predictions_valid_replicate0_pytorch.csv
Total genes (forward+reverse): 3038


 46%|████▌     | 1402/3038 [16:06<18:40,  1.46it/s]

Gene has empty output slice: ENSG00000178591.6


 46%|████▌     | 1403/3038 [16:06<18:38,  1.46it/s]

Gene has empty output slice: ENSG00000125788.5


...

 87%|████████▋ | 2642/3038 [30:22<04:32,  1.45it/s]

Gene has empty output slice: ENSG00000249054.2


100%|██████████| 3038/3038 [34:55<00:00,  1.45it/s]

Done: (3032, 14)
Missing genes count: 0
/home/jovyan/shares/SR003.nfs2/aspeedok/GENA_LM/downstream_tasks/expression_prediction/benchmarks/borzoi/borzoi_predictions_valid_replicate0_pytorch.csv
p = 0


experiment_id,gene_id,ENCSR045GTF,ENCSR094GVZ,ENCSR128CYL,ENCSR245ATJ,ENCSR357BYU,ENCSR432EBE,ENCSR471RUK,ENCSR561FEE,ENCSR571BML,ENCSR586SYA,ENCSR721HDG,ENCSR763OMY,ENCSR892LBU
0,ENSG00000001617.11,9.003489,2.208751,10.638779,7.383554,2.487712,5.515182,2.852123,3.140740,8.190046,4.473660,3.051635,3.702394,9.682981
1,ENSG00000002016.17,7.369006,5.286678,12.037867,9.629592,4.296402,4.341439,3.848074,6.057591,5.213218,4.468218,2.641327,7.007655,7.390925
2,ENSG00000002549.12,35.790066,26.572323,42.150757,48.812397,45.503750,25.840006,18.658279,37.664036,32.931114,20.858244,22.286028,50.044952,36.992836
3,ENSG00000002587.9,2.180358,0.962996,1.024214,0.090757,0.147700,0.589315,0.722051,0.033116,0.827947,0.863152,0.396425,0.821109,1.989991
4,ENSG00000003393.14,9.876023,7.996849,13.640111,11.737370,8.963378,7.509901,6.193588,7.341879,5.567631,8.082061,2.208677,13.522211,11.963784


In [23]:
bw_df_f = pd.read_csv(genes_forward_path, sep="\t")
bw_df_r = pd.read_csv(genes_reverse_path, sep="\t")
bw_df = pd.concat([bw_df_f, bw_df_r], ignore_index=True)
bw_df

,chrom,start,end,transcript_id,gene_strand,gene_id,gene_name,chromosome,gene_start,gene_end,transcript_name,transcript_start,transcript_end,TSS,TES,gene_id_unversioned,split
0,chr2,52864235,52864235,ENST00000418451.1,+,ENSG00000232604.1,AC010967.2,chr2,52864235,52866631,AC010967.2-201,52864235,52866631,52864235,52866631,ENSG00000232604,valid
1,chr2,53767792,53767792,ENST00000295304.4,+,ENSG00000143942.4,CHAC2,chr2,53767792,53775196,CHAC2-201,53767792,53775196,53767792,53775196,ENSG00000143942,valid
2,chr2,53787080,53787080,ENST00000185150.8,+,ENSG00000068912.13,ERLEC1,chr2,53787044,53818819,ERLEC1-201,53787080,53818819,53787080,53818819,ENSG00000068912,valid
3,chr2,54115403,54115403,ENST00000406041.5,+,ENSG00000170634.12,ACYP2,chr2,53970838,54305300,ACYP2-203,54115403,54143314,54115403,54143314,ENSG00000170634,valid
4,chr2,54082554,54082554,ENST00000606436.1,+,ENSG00000272156.1,AC008280.3,chr2,54082554,54085066,AC008280.3-201,54082554,54085066,54082554,54085066,ENSG00000272156,valid
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3033,chrX,8801383,8801383,ENST00000381003.7,-,ENSG00000183304.10,FAM9A,chrX,8790795,8801383,FAM9A-201,8790795,8801383,8801383,8790795,ENSG00000183304,valid
3034,chrX,8867601,8867601,ENST00000440430.1,-,ENSG00000229012.1,AC003685.1,chrX,8863861,8867601,AC003685.1-201,8863861,8867601,8867601,8863861,ENSG00000229012,valid
3035,chrX,8927153,8927153,ENST00000650341.1,-,ENSG00000285896.1,AC074281.2,chrX,8897642,8927153,AC074281.2-201,8897642,8927153,8927153,8897642,ENSG00000285896,valid
3036,chrX,9034127,9034127,ENST00000327220.9,-,ENSG00000177138.15,FAM9B,chrX,9024232,9164639,FAM9B-201,9024995,9034127,9034127,9024995,ENSG00000177138,valid


## Шаг 2. Benchmark / метрики

Здесь объединяется логика из `only_benchmarking_clean.ipynb`:

- выравнивание `true` и `pred`
- `score_predictions(...)`
- средняя корреляция по генам
- средняя корреляция по cell types


In [24]:
if SCORE_SRC_DIR not in sys.path:
    sys.path.append(SCORE_SRC_DIR)

from score_ct_specificity import score_predictions


def align_dataframes(true_df, pred_df):
    true_subset = true_df[true_df["gene_id"].isin(pred_df["gene_id"])].reset_index(drop=True)
    pred_subset = pred_df[pred_df["gene_id"].isin(true_subset["gene_id"])].reset_index(drop=True)
    true_sorted = true_subset.sort_values("gene_id")
    pred_sorted = pred_subset.sort_values("gene_id")
    common_cols = sorted(set(true_df.columns[1:]).intersection(pred_df.columns[1:]))
    true_aligned = true_sorted[["gene_id"] + common_cols]
    pred_aligned = pred_sorted[["gene_id"] + common_cols]
    return true_aligned, pred_aligned


map_df = pd.read_csv(MAPPING_PATH)
encff_to_encsr = dict(zip(map_df["original_id"], map_df["id"]))

pred_eval_df = pred_gene_df.rename(columns=encff_to_encsr)

true_df_main = pd.read_csv(true_main_path).rename(columns=encff_to_encsr)
# true_df_tcell = pd.read_csv(true_tcell_bw_path).rename(columns=encff_to_encsr)
# if "ENCFF033KAE" in true_df_tcell.columns:
#     true_df_tcell["ENCFF033KAE"] = np.log1p(true_df_tcell["ENCFF033KAE"])

if true_source == "main":
    true_eval_df = true_df_main
elif true_source == "tcell_bw":
    true_eval_df = true_df_tcell
else:
    raise ValueError(f"Unknown true_source: {true_source}")

print("Prediction shape:", pred_eval_df.shape)
print("Truth shape:", true_eval_df.shape)


Prediction shape: (3032, 14)
Truth shape: (3038, 15)


In [48]:
pred_eval_df

experiment_id,gene_id,ENCFF242BWW,ENCFF035CWS,ENCFF602HCV,ENCFF660EXG,ENCFF784MDF,ENCFF664WLU,ENCFF857JQM,ENCFF761SPP,ENCFF494KRC,ENCFF083EOC,ENCFF361XCF,ENCFF236XOK,ENCFF123KIW
0,ENSG00000003137.8,1.444014,1.119174,0.853617,0.396988,0.477500,0.753891,0.540777,0.081037,0.384880,0.747582,0.148234,1.422443,1.699554
1,ENSG00000005436.13,0.029933,0.013620,0.035675,0.041604,0.019346,0.011983,0.010517,0.012000,0.008925,0.018019,0.001385,0.010954,0.019120
2,ENSG00000005448.16,5.541739,1.239172,5.963140,4.981400,3.081889,3.707099,1.953269,2.144186,7.028134,2.815510,2.533532,2.850324,4.821231
3,ENSG00000034510.5,152.426376,111.095352,132.962921,56.585815,45.360226,55.167625,49.890778,32.370499,109.440422,43.250015,85.708649,168.819565,127.697594
4,ENSG00000035141.7,0.022343,0.007086,0.028546,0.022671,0.015595,0.015143,0.005623,0.004629,0.016482,0.015702,0.007706,0.006386,0.014410
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
164,ENSG00000251370.1,0.505920,0.592998,0.066946,0.029811,0.085258,0.555261,0.495170,0.001162,0.017328,1.042189,0.001642,0.162138,0.654281
165,ENSG00000260840.1,0.002670,0.001398,0.001374,0.001775,0.001997,0.002128,0.000979,0.000212,0.003731,0.002132,0.000627,0.000506,0.002046
166,ENSG00000263590.2,0.078277,0.017414,0.056132,0.048008,0.049919,0.051063,0.015149,0.012904,0.050546,0.040777,0.009153,0.017947,0.057149
167,ENSG00000274049.4,24.551085,12.860794,34.671867,24.866875,17.906277,12.684768,7.529543,15.531938,27.085045,7.513259,16.647699,31.858788,21.885828


In [25]:
true_aligned, pred_aligned = align_dataframes(true_eval_df, pred_eval_df)
value_cols = pred_aligned.columns[1:]
pred_aligned_copy = pred_aligned.copy()
pred_aligned[value_cols] = np.log10(pred_aligned[value_cols] + 1)

# score_dict = score_predictions(
#     true_aligned,
#     pred_aligned,
#     SELECTED_TARGETS_PATH,
#     return_big_data=True,
# )
# deviation_r = score_dict.get("deviation_r", float("nan"))

# Корреляция по строкам: для каждого gene_id сравниваем профиль по всем таргетам.
cell_corrs = []
for i in range(true_aligned.shape[0]):
    true_vec = np.array(true_aligned.iloc[i, 1:].values, dtype=np.float64)
    pred_vec = np.array(pred_aligned.iloc[i, 1:].values, dtype=np.float64)
    if np.std(true_vec) == 0 or np.std(pred_vec) == 0:
        continue
    cell_corrs.append(np.corrcoef(true_vec, pred_vec)[0, 1])
avg_cell_corr = float(np.nan) if len(cell_corrs) == 0 else float(np.mean(cell_corrs))

# Корреляция по колонкам: для каждого таргета сравниваем профиль по всем генам.
gene_corrs = []
for j in range(1, true_aligned.shape[1]):
    true_vec = np.array(true_aligned.iloc[:, j].values, dtype=np.float64)
    pred_vec = np.array(pred_aligned.iloc[:, j].values, dtype=np.float64)
    if np.std(true_vec) == 0 or np.std(pred_vec) == 0:
        continue
    gene_corrs.append(np.corrcoef(true_vec, pred_vec)[0, 1])
avg_gene_corr = float(np.nan) if len(gene_corrs) == 0 else float(np.mean(gene_corrs))

summary_df = pd.DataFrame([
    {
        "split": split,
        "model_variant": model_variant,
        "model_name": model_name,
        "true_source": true_source,
        "autocast_enabled": use_autocast,
        "avg_gene_corr": avg_gene_corr,
        "avg_cell_type_corr": avg_cell_corr,
        "genes_evaluated": len(cell_corrs),
        "cell_types_evaluated": len(gene_corrs),
    }
])

summary_df


,split,model_variant,model_name,true_source,autocast_enabled,avg_gene_corr,avg_cell_type_corr,genes_evaluated,cell_types_evaluated
0,valid,borzoi,johahi/borzoi-replicate-0,main,False,0.763692,0.162749,3032,13


In [51]:
value_cols = pred_aligned.columns[1:]
pred_aligned_copy = pred_aligned.copy()
pred_aligned[value_cols] = np.log10(pred_aligned[value_cols] + 1)


In [52]:
pred_aligned

experiment_id,gene_id,ENCFF035CWS,ENCFF083EOC,ENCFF123KIW,ENCFF236XOK,ENCFF242BWW,ENCFF361XCF,ENCFF494KRC,ENCFF602HCV,ENCFF660EXG,ENCFF664WLU,ENCFF761SPP,ENCFF784MDF,ENCFF857JQM
0,ENSG00000003137.8,0.326167,0.242438,0.431292,0.384254,0.388104,0.060030,0.141412,0.268020,0.145193,0.244003,0.033841,0.169527,0.187740
1,ENSG00000005436.13,0.005875,0.007756,0.008225,0.004731,0.012809,0.000601,0.003859,0.015223,0.017703,0.005173,0.005181,0.008322,0.004544
2,ENSG00000005448.16,0.350088,0.581553,0.765015,0.585497,0.815693,0.548209,0.904615,0.842805,0.776803,0.672753,0.497508,0.610861,0.470303
3,ENSG00000034510.5,2.049588,1.645913,2.109571,2.229988,2.185900,1.938062,2.043128,2.126985,1.760316,1.749486,1.523363,1.666146,1.706639
4,ENSG00000035141.7,0.003067,0.006767,0.006214,0.002765,0.009597,0.003334,0.007100,0.012224,0.009736,0.006527,0.002006,0.006721,0.002435
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
164,ENSG00000251370.1,0.202215,0.310096,0.218609,0.065258,0.177802,0.000712,0.007461,0.028143,0.012758,0.191803,0.000505,0.035533,0.174691
165,ENSG00000260840.1,0.000607,0.000925,0.000888,0.000220,0.001158,0.000272,0.001617,0.000596,0.000770,0.000923,0.000092,0.000866,0.000425
166,ENSG00000263590.2,0.007498,0.017358,0.024136,0.007725,0.032730,0.003957,0.021415,0.023718,0.020365,0.021629,0.005568,0.021156,0.006530
167,ENSG00000274049.4,1.141788,0.930096,1.359567,1.516652,1.407409,1.246688,1.448475,1.552326,1.412744,1.136237,1.218324,1.276606,0.930926


In [53]:
true_aligned

,gene_id,ENCFF035CWS,ENCFF083EOC,ENCFF123KIW,ENCFF236XOK,ENCFF242BWW,ENCFF361XCF,ENCFF494KRC,ENCFF602HCV,ENCFF660EXG,ENCFF664WLU,ENCFF761SPP,ENCFF784MDF,ENCFF857JQM
0,ENSG00000003137.8,0.190620,0.019803,0.587787,1.800058,0.615186,0.157004,0.104360,1.752672,0.095310,0.131028,0.148420,1.007958,0.000000
1,ENSG00000005436.13,1.150572,0.746688,2.250239,1.896119,1.798404,1.011601,2.316488,2.419479,2.613739,1.406097,3.519573,2.086914,0.993252
2,ENSG00000005448.16,0.593327,0.438255,2.128232,1.867176,2.694627,0.989541,2.371178,2.468100,0.667829,0.506818,0.727549,0.476234,0.398776
3,ENSG00000034510.5,4.866380,3.292126,6.239476,5.346536,6.403524,6.339212,5.522780,6.616293,5.617462,3.808439,6.098906,3.821661,4.928267
4,ENSG00000035141.7,0.932164,1.040277,2.260721,2.192770,2.429218,2.000128,3.124125,3.082369,3.656615,1.940179,4.638702,2.366498,1.244155
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
164,ENSG00000251370.1,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
165,ENSG00000260840.1,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
166,ENSG00000263590.2,0.000000,0.000000,0.039221,0.000000,0.000000,0.000000,0.000000,0.009950,0.000000,0.029559,0.000000,0.000000,0.104360
167,ENSG00000274049.4,0.457425,0.086178,0.254642,0.000000,0.000000,0.598836,0.000000,0.000000,0.000000,0.000000,2.064328,0.000000,0.277632


## Шаг 3. Scatter plots

По умолчанию рисуются все общие таргеты. Если это слишком много, задай `plot_targets = [...]`
в конфигурационной ячейке выше и перезапусти plotting cell.


In [1]:
targets_for_plot = [c for c in true_aligned.columns if c != "gene_id"]
if plot_targets is not None:
    targets_for_plot = [t for t in plot_targets if t in targets_for_plot]

print("Targets to plot:", len(targets_for_plot))

for target in targets_for_plot:
    x = pred_aligned[target].values
    y = true_aligned[target].values

    if np.std(x) == 0 or np.std(y) == 0:
        print(f"Skip {target}: zero variance")
        continue

    p = np.corrcoef(x, y)[0, 1]

    plt.figure(figsize=(4, 4))
    plt.scatter(x, y, s=6, alpha=0.6)
    plt.xlabel("pred")
    plt.ylabel("true")
    plt.title(f"{target} | Pearson={p:.3f}")
    plt.plot([x.min(), x.max()], [x.min(), x.max()], "k--", alpha=0.3)
    plt.tight_layout()
    plt.show()


NameError: name 'true_aligned' is not defined